# Triagem de cinco classes: candidatos C0 a C4

Protocolo em `triagem_README.md` (seção 1, com as revisões R1 a R3 na 1.11).
Todas as métricas usam os logits fora da dobra gravados na pasta de trabalho.
As saídas mostram só contagens, pseudônimos e números.

## 1. Dados

In [2]:
import time
import warnings

import numpy as np
import pandas as pd

from meter_reader.config import get_work_dir
from meter_reader.contracts import PRODUCT_CLASSES
from training.build_manifest import file_sha256
from training.group_dataset import DEV_LABELS, check_group_sizes, load_group_photos
from training.group_labels import TRAINING_GROUP_CLASS
from training.triage_eval import BROAD_CLASS, OOF_FOLDER, load_oof, save_oof
from training.triage_features import cache_path, load_or_extract

WORK_DIR = get_work_dir()
CACHE_DIR = WORK_DIR / "caracteristicas"
LABELS_SHA = DEV_LABELS[1]

photos = load_group_photos()  # confere o SHA-256 dos rótulos (protocolo 1.1)
codes = [p.pseudonym for p in photos]
print("Fotos rotuladas:", len(photos), "| pareadas:", sum(p.paired for p in photos))
print("Grupos de treino:", check_group_sizes(photos))
for name in ("mobilenet_v3_large_224", "dinov2_224"):
    print(f"Cache {name}: SHA-256 {file_sha256(cache_path(CACHE_DIR, name, 5))}")

Fotos rotuladas: 9301 | pareadas: 8637
Grupos de treino: {'dig1_quadrado_branco': 6372, 'dig2_redondo_branco': 208, 'dig3_vidro_redondo': 369, 'dig4_verde_arduino': 76, 'cic_disco': 616, 'cic_duas_leituras': 173, 'cic_uma_leitura': 658, 'indeterminado': 255, 'multiplos': 72, 'out_acesso': 307, 'out_sem_acesso': 105, 'out_sem_contexto': 90}
Cache mobilenet_v3_large_224: SHA-256 c44b4f96623d3574482f351141125aa98e723d9b0dd81565d17e68b78a35a8d1
Cache dinov2_224: SHA-256 30e002f5feacf1ba58d17fdfaaf9247b7b6eb04cd379daa566b303a93ab0b28f


## 2. C0: MobileNetV3-Large congelada e sonda linear

Três execuções: W-b com rótulo fino (padrão), W-d com rótulo fino (ablação de
pesos, 1.4) e W-b com rótulo amplo (ablação fino × amplo, 1.5). Uma execução
por dobra (revisão R1). Abaixo, a evidência da R1 nesta base.

In [3]:
from sklearn.linear_model import LogisticRegression

c0_feats = load_or_extract([p.path for p in photos], codes, "mobilenet_v3_large_224", 5, CACHE_DIR)
rng = np.random.default_rng(0)
sub = rng.choice(len(photos), size=1500, replace=False)  # subamostra: só a evidência
x_sub = c0_feats[sub, 0]
x_sub = (x_sub - x_sub.mean(axis=0)) / (x_sub.std(axis=0) + 1e-6)
y_sub = np.array([photos[i].target for i in sub])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a subamostra pode não convergir; aqui só importa a igualdade
    a = LogisticRegression(max_iter=2000, random_state=0).fit(x_sub, y_sub).coef_
    b = LogisticRegression(max_iter=2000, random_state=99).fit(x_sub, y_sub).coef_
print("Coeficientes idênticos com random_state 0 e 99:", np.array_equal(a, b))

Coeficientes idênticos com random_state 0 e 99: True


In [4]:
from training.linear_probe import cross_validate_probe

EVAL_CODES = None  # ordem das pareadas avaliadas, igual em todas as execuções


def run_probe(name, feats, targets, output_class, weighting, config):
    """Treina e grava os logits fora da dobra; se já gravados, só lê."""
    global EVAL_CODES
    if (WORK_DIR / OOF_FOLDER / f"{name}.npz").exists():
        data = np.load(WORK_DIR / OOF_FOLDER / f"{name}.npz")
        EVAL_CODES = EVAL_CODES or list(data["codes"])
        print(f"{name}: lido do arquivo gravado")
        return load_oof(WORK_DIR, name, EVAL_CODES)
    start = time.perf_counter()
    logits, index, folds = cross_validate_probe(feats, photos, targets, output_class, weighting)
    eval_codes = [codes[i] for i in index]
    EVAL_CODES = EVAL_CODES or eval_codes
    assert eval_codes == EVAL_CODES, "Ordem das fotos avaliadas diferente entre execuções."
    save_oof(WORK_DIR, name, eval_codes, [photos[i].lot for i in index],
             logits[None], {**config, "ponderacao": weighting,
                            "dobras": [vars(f) for f in folds]}, LABELS_SHA)
    minutes = (time.perf_counter() - start) / 60
    print(f"{name}: {minutes:.1f} min; iterações por dobra {[f.n_iter for f in folds]}")
    return logits


fine = np.array([p.target for p in photos])
broad = np.array([p.product_class for p in photos])
base = {"candidato": "C0", "extrator": "mobilenet_v3_large_224", "vistas": 5}

oof = {
    "C0 W-b fino": run_probe("c0_wb_fino", c0_feats, fine, TRAINING_GROUP_CLASS, "W-b",
                             {**base, "rotulo": "fino"}),
    "C0 W-d fino": run_probe("c0_wd_fino", c0_feats, fine, TRAINING_GROUP_CLASS, "W-d",
                             {**base, "rotulo": "fino"}),
    "C0 W-b amplo": run_probe("c0_wb_amplo", c0_feats, broad, BROAD_CLASS, "W-b",
                              {**base, "rotulo": "amplo"}),
}

c0_wb_fino: 0.8 min; iterações por dobra [128, 133, 123]
c0_wd_fino: 0.5 min; iterações por dobra [123, 105, 108]
c0_wb_amplo: 0.5 min; iterações por dobra [145, 139, 187]
